# 20 — Technique: lexicon correction

**Source.** Senevirathna et al. (2025), *Enhancing Multilingual Sentiment Analysis with
Explainability for Sinhala, English, and Code-Mixed Content* (arXiv:2504.13545) — the single most
on-task paper in `research/`: banking-domain sentiment over Sinhala, Singlish and code-mixed text.

Their pipeline is XLM-R **plus a domain-specific sentiment lexicon applied as a post-processing
correction**, combined by softmax-weighted aggregation. The lexicon layer alone was worth
**+10.2pp accuracy / +0.10 F1**:

| model | accuracy | F1 |
|---|---|---|
| SVM | 62.4% | 0.58 |
| XLM-R (fine-tuned) | 78.2% | 0.74 |
| GPT-4o (zero-shot) | 81.5% | 0.77 |
| **XLM-R + lexicon correction** | **88.4%** | **0.84** |

Their stated motivation is that banking-Singlish sentiment terms — `"app eka lag wenawa"`,
`"godak slow"` — are missed by the pretrained model and only caught after lexicon correction.

---

## One honest difference, stated up front

**Their lexicon was authored externally; ours is mined from the training data.** There is no
polarity lexicon in this repo — `SINHALA_STYLE.md` is a *translation* glossary (vocabulary and
register), not sentiment. Hand-authoring one across five language tracks is a labeling project,
not a notebook.

So this notebook mines the lexicon from the **train split only**, by log-odds ratio with an
informative Dirichlet prior (Monroe, Colaresi & Quinn 2008) — the standard estimator for
class-associated terms when one class is small.

That difference matters and caps the expected gain: **a lexicon derived from the same rows the
classifier already trained on carries little information the classifier has not already
extracted.** Senevirathna's lexicon was outside knowledge; ours is a re-reading of the inside.
If this technique is going to help here, it will be by a much smaller margin than +10.2pp — and
a null result is a real finding, not a failed run. It would say the +10.2pp came from the
*externality* of the lexicon, not from the correction mechanism.

**A tokenization defect found while building this.** sklearn's default `token_pattern` drops
Unicode combining marks, so every Sinhala and Tamil vowel sign was being stripped — the first run
of this notebook mined `කව හර` instead of `කවුරු හරි`. This notebook now tokenizes through
`swiftbench.tokenize` (indic-nlp, script-dispatched), and `models.py` has been switched to the
same. Full comparison in [`08_word_tokenizer_comparison.ipynb`](08_word_tokenizer_comparison.ipynb).

**Protocol.** Lexicon mined on train. Blend weight and threshold tuned by 5-fold CV *within
train*. Evaluated once on dev. Test is not opened.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import StratifiedKFold

import swiftbench as sb
from swiftbench import config, data, imbalance, metrics, models, splits, tokenize as sbtok

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

AUTHOR = "sithija"
LANGS = config.LANGUAGES
POS = config.SENTIMENT_POSITIVE_CLASS      # "Negative"

# Tokenization goes through swiftbench.tokenize (indic-nlp, script-dispatched). sklearn's
# default drops Sinhala/Tamil vowel signs and a bare regex breaks ZWJ conjuncts -- see
# 08_word_tokenizer_comparison.ipynb. The first version of this notebook mined the mangled
# stem "kav hari" instead of the real word.
TOKENIZER = sbtok.tokenize

train = splits.get(LANGS, "train")
dev = splits.get(LANGS, "dev")
print("split sha:", splits.sha())
print(f"train {len(train):,} rows   dev {len(dev):,} rows")
print("train Negative rows:", int((train.sentiment == POS).sum()))

split sha: e7b5934392cd
train 42,500 rows   dev 7,490 rows
train Negative rows: 1970


## 1. Mine the lexicon

Log-odds ratio with an informative Dirichlet prior. For term $w$ in class $i$:

$$\hat\delta_w = \log\frac{y_w^i + \alpha_w}{n^i + \alpha_0 - y_w^i - \alpha_w} - \log\frac{y_w^j + \alpha_w}{n^j + \alpha_0 - y_w^j - \alpha_w}$$

with the prior $\alpha_w$ taken from the corpus-wide frequency, and each $\hat\delta_w$ divided by
its own standard error to give a z-score. Raw frequency ratios are hopeless for a 4.6%-prevalence
class — a term appearing twice, both times in Negative rows, would score infinitely negative.
The prior shrinks exactly those terms.

Mined **per language**, because Singlish `"godak slow"` and Sinhala `"හරිම හෙමින්"` are different
strings for the same complaint and pooling them would dilute both.

In [2]:
def mine_lexicon(df, lang, top_k=60, min_count=5, ngram=(1, 2)):
    # Log-odds-with-prior z-scores for Negative-vs-Neutral terms in one language.
    sub = df[df.language == lang]
    vec = CountVectorizer(ngram_range=ngram, min_df=min_count, lowercase=True,
                          tokenizer=TOKENIZER, token_pattern=None)
    X = vec.fit_transform(sub[config.TEXT_COLUMN])
    terms = np.array(vec.get_feature_names_out())

    is_neg = (sub.sentiment == POS).to_numpy()
    y_neg = np.asarray(X[is_neg].sum(0)).ravel().astype(float)
    y_neu = np.asarray(X[~is_neg].sum(0)).ravel().astype(float)

    a_w = (y_neg + y_neu)                      # corpus frequency as the prior
    a_0 = a_w.sum()
    n_neg, n_neu = y_neg.sum(), y_neu.sum()

    def lo(y, n):
        num = y + a_w
        den = n + a_0 - y - a_w
        return np.log(num / den), 1.0 / num     # log-odds and its variance term

    l_neg, v_neg = lo(y_neg, n_neg)
    l_neu, v_neu = lo(y_neu, n_neu)
    z = (l_neg - l_neu) / np.sqrt(v_neg + v_neu)

    order = np.argsort(-z)[:top_k]
    return pd.DataFrame({"language": lang, "term": terms[order], "z": z[order],
                         "n_negative": y_neg[order].astype(int),
                         "n_neutral": y_neu[order].astype(int)})

lexicon = pd.concat([mine_lexicon(train, l) for l in LANGS], ignore_index=True)
print(f"{len(lexicon)} terms mined ({lexicon.language.nunique()} languages)")
for l in LANGS:
    top = lexicon[lexicon.language == l].head(8)
    print(f"\n{l}:")
    print("   " + " | ".join(f"{t} ({z:.1f})" for t, z in zip(top.term, top.z)))

300 terms mined (5 languages)

english:
   someone has (3.7) | really (3.6) | someone (3.5) | now (3.4) | you keep (3.1) | and (2.9) | didn do (2.9) | ve (2.9)

sinhala:
   කවුරු හරි (3.8) | කවුරු (3.8) | හරි මගේ (3.4) | කරන්නේ නැති (3.0) | මේක (3.0) | මම කරන්නේ (3.0) | දැන්ම (2.9) | wallet එක (2.9)

singlish:
   kavuru (3.8) | kavuru hari (3.8) | hari mage (3.4) | karanne nati (3.0) | meka (3.0) | mama karanne (3.0) | danma (2.9) | wallet eka (2.9)

tamil:
   மிகவும் (4.5) | இது (3.6) | செய்யாத (3.6) | யாரோ (3.4) | நான் செய்யாத (3.3) | தொடர்ந்து (3.1) | இப்போது (3.1) | போய்விட்டது (3.0)

tamilish:
   romba (4.9) | poiduchu (4.5) | yaaro (4.1) | thadava try (3.2) | ithu (3.0) | kaanamal (2.8) | 500 (2.8) | 500 000 (2.8)


In [3]:
display(lexicon.groupby("language").head(5).reset_index(drop=True))
lexicon.to_csv(REPO / "ml" / "reports" / "sentiment_lexicon.csv", index=False)
print("wrote ml/reports/sentiment_lexicon.csv")

,language,term,z,n_negative,n_neutral
0,english,someone has,3.7066,19,9
1,english,really,3.5938,26,26
2,english,someone,3.5416,49,105
3,english,now,3.3885,57,149
4,english,you keep,3.1071,9,0
5,sinhala,කවුරු හරි,3.8229,53,110
6,sinhala,කවුරු,3.8229,53,110
7,sinhala,හරි මගේ,3.3889,33,54
8,sinhala,කරන්නේ නැති,2.9826,30,58
9,sinhala,මේක,2.9820,50,149


wrote ml/reports/sentiment_lexicon.csv


## 2. The correction layer

Senevirathna combine model and lexicon by softmax-weighted aggregation. Implemented here as a
blend of two z-scored signals, which is the same idea with one tunable knob:

```
score = z(model_decision) + alpha * z(lexicon_hits)
```

`alpha = 0` recovers the model untouched, so the ablation is exact rather than approximate. Both
signals are standardised using **train** statistics only, so dev never informs the scaling.

In [4]:
def lexicon_score(texts, langs, lex):
    # Sum of z-weights of lexicon terms present, per row.
    by_lang = {l: g for l, g in lex.groupby("language")}
    out = np.zeros(len(texts))
    for lang, g in by_lang.items():
        m = (langs == lang).to_numpy()
        if not m.any():
            continue
        vec = CountVectorizer(vocabulary=list(g.term), ngram_range=(1, 2), lowercase=True,
                              tokenizer=TOKENIZER, token_pattern=None)
        hits = vec.transform(pd.Series(texts)[m])
        out[m] = hits.dot(g.z.to_numpy())
    return out


def model_score(pipe, texts):
    est, feats = pipe[-1], pipe[:-1]
    Z = feats.transform(texts)
    if hasattr(est, "predict_proba"):
        return est.predict_proba(Z)[:, list(est.classes_).index(POS)]
    s = est.decision_function(Z)
    return s if list(est.classes_)[1] == POS else -s


def fit_champion(df):
    fit = imbalance.resample(df, "sentiment", "class_weight")
    clf = models.build("tfidf-svm", class_weight="balanced", C=0.5)
    clf.fit(fit[config.TEXT_COLUMN], fit.sentiment)
    return clf

## 3. Tune `alpha` and the threshold — inside train only

5-fold CV over train ids. Each fold re-mines the lexicon on its own training portion, so the
lexicon never sees the fold it is scored on. Skipping that would leak: terms mined from a row are
near-guaranteed to fire on it.

In [5]:
ALPHAS = [0.0, 0.05, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0, 1.5]

train_ids = sorted(train.id.unique())
eng = data.load_language("english", "train").set_index("id")
strat = eng.loc[train_ids, "sentiment"].values
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.RANDOM_STATE)

rows = []
for fold, (a, b) in enumerate(kf.split(train_ids, strat)):
    tr_ids, va_ids = {train_ids[i] for i in a}, {train_ids[i] for i in b}
    tr = train[train.id.isin(tr_ids)]
    va = train[train.id.isin(va_ids)]

    clf = fit_champion(tr)
    lex = pd.concat([mine_lexicon(tr, l) for l in LANGS], ignore_index=True)

    ms_tr = model_score(clf, tr[config.TEXT_COLUMN]); ls_tr = lexicon_score(tr[config.TEXT_COLUMN], tr.language, lex)
    mu_m, sd_m = ms_tr.mean(), ms_tr.std() + 1e-9
    mu_l, sd_l = ls_tr.mean(), ls_tr.std() + 1e-9

    ms = (model_score(clf, va[config.TEXT_COLUMN]) - mu_m) / sd_m
    ls = (lexicon_score(va[config.TEXT_COLUMN], va.language, lex) - mu_l) / sd_l
    yt = va.sentiment.to_numpy()

    for alpha in ALPHAS:
        s = ms + alpha * ls
        grid = np.quantile(s, np.linspace(0.80, 0.999, 120))
        best_f1, best_t = -1.0, None
        for t in grid:
            f = metrics.score(yt, np.where(s >= t, POS, "Neutral"), "sentiment")["negative_f1"]
            if f > best_f1:
                best_f1, best_t = f, float(t)
        rows.append({"fold": fold, "alpha": alpha, "negative_f1": best_f1, "threshold": best_t})
    print(f"  fold {fold} done", flush=True)

cv = pd.DataFrame(rows)
summary = cv.groupby("alpha").agg(mean_f1=("negative_f1", "mean"), std_f1=("negative_f1", "std"),
                                  mean_t=("threshold", "mean")).reset_index()
display(summary)

best_alpha = float(summary.loc[summary.mean_f1.idxmax(), "alpha"])
gain = summary.mean_f1.max() - summary.loc[summary.alpha == 0.0, "mean_f1"].iloc[0]
print(f"\nbest alpha = {best_alpha}   CV gain over alpha=0 (model alone): {gain:+.4f}")

  fold 0 done


  fold 1 done


  fold 2 done


  fold 3 done


  fold 4 done


,alpha,mean_f1,std_f1,mean_t
0,0.0000,0.5554,0.0250,1.7693
1,0.0500,0.5544,0.0225,1.8145
2,0.1000,0.5540,0.0249,1.8411
3,0.2000,0.5489,0.0259,1.8758
4,0.3500,0.5307,0.0284,1.9588
5,0.5000,0.5106,0.0245,2.1364
6,0.7500,0.4864,0.0209,2.4302
7,1.0000,0.4697,0.0203,2.7524
8,1.5000,0.4438,0.0177,3.3491



best alpha = 0.0   CV gain over alpha=0 (model alone): +0.0000


## 4. The ablation, on dev

`alpha = 0` versus the CV-chosen `alpha`, same model, same threshold procedure. CI resampled over
ticket `id`.

In [6]:
clf = fit_champion(train)
lex_full = lexicon
ms_tr = model_score(clf, train[config.TEXT_COLUMN])
ls_tr = lexicon_score(train[config.TEXT_COLUMN], train.language, lex_full)
mu_m, sd_m = ms_tr.mean(), ms_tr.std() + 1e-9
mu_l, sd_l = ls_tr.mean(), ls_tr.std() + 1e-9

ms_dev = (model_score(clf, dev[config.TEXT_COLUMN]) - mu_m) / sd_m
ls_dev = (lexicon_score(dev[config.TEXT_COLUMN], dev.language, lex_full) - mu_l) / sd_l
yt_dev = dev.sentiment.to_numpy()

def boot(y_true, y_pred, ids, n_boot=1000, seed=42):
    yt = (np.asarray(y_true) == POS); yp = (np.asarray(y_pred) == POS)
    groups = [g.to_numpy() for _, g in pd.Series(np.arange(len(yt))).groupby(np.asarray(ids))]
    rng = np.random.default_rng(seed); n = len(groups); out = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([groups[j] for j in rng.integers(0, n, n)])
        tp = (yt[idx] & yp[idx]).sum(); fp = (~yt[idx] & yp[idx]).sum(); fn = (yt[idx] & ~yp[idx]).sum()
        out[i] = 0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn)
    return np.percentile(out, [2.5, 97.5])

res = []
nonzero = summary[summary.alpha > 0]
best_nonzero = float(nonzero.loc[nonzero.mean_f1.idxmax(), "alpha"])
arms = [("model only (alpha=0)", 0.0), (f"model + lexicon (best non-zero alpha={best_nonzero})", best_nonzero)]
if best_alpha not in (0.0, best_nonzero):
    arms.append((f"model + lexicon (CV alpha={best_alpha})", best_alpha))

for name, alpha in arms:
    t = float(summary.loc[summary.alpha == alpha, "mean_t"].iloc[0])
    s = ms_dev + alpha * ls_dev
    pred = np.where(s >= t, POS, "Neutral")
    sc = metrics.score(yt_dev, pred, "sentiment")
    lo, hi = boot(yt_dev, pred, dev.id.to_numpy())
    res.append({"setup": name, "alpha": alpha, "threshold": t,
                "negative_f1": sc["negative_f1"], "precision": sc["negative_precision"],
                "recall": sc["negative_recall"], "accuracy": sc["accuracy"],
                "ci_low": lo, "ci_high": hi})

ablation = pd.DataFrame(res)
display(ablation)

delta = ablation.negative_f1.iloc[1] - ablation.negative_f1.iloc[0]
print(f"\nlexicon layer contributes {delta:+.4f} negative_f1 on dev")
print(f"Senevirathna et al. report +0.10 F1 for an externally authored lexicon.")

,setup,alpha,threshold,negative_f1,precision,recall,accuracy,ci_low,ci_high
0,model only (alpha=0),0.0000,1.7693,0.6173,0.5784,0.6618,0.9628,0.5402,0.6919
1,model + lexicon (best non-zero alpha=0.05),0.0500,1.8145,0.6098,0.5653,0.6618,0.9615,0.5320,0.6824



lexicon layer contributes -0.0075 negative_f1 on dev
Senevirathna et al. report +0.10 F1 for an externally authored lexicon.


In [7]:
# Per-language -- the paper's claim is specifically about Sinhala/Singlish code-mixed terms.
rows = []
for lang in LANGS:
    m = (dev.language == lang).to_numpy()
    for name, alpha in [("model", 0.0), ("model+lexicon", best_nonzero)]:
        t = float(summary.loc[summary.alpha == alpha, "mean_t"].iloc[0])
        pred = np.where((ms_dev + alpha * ls_dev)[m] >= t, POS, "Neutral")
        sc = metrics.score(yt_dev[m], pred, "sentiment")
        rows.append({"language": lang, "setup": name, "negative_f1": sc["negative_f1"],
                     "precision": sc["negative_precision"], "recall": sc["negative_recall"]})
per_lang = pd.DataFrame(rows).pivot(index="language", columns="setup", values="negative_f1")
per_lang["delta"] = per_lang["model+lexicon"] - per_lang["model"]
display(per_lang)

ablation.to_csv(REPO / "ml" / "reports" / "technique_lexicon_ablation.csv", index=False)
per_lang.to_csv(REPO / "ml" / "reports" / "technique_lexicon_by_language.csv")
print("wrote technique_lexicon_ablation.csv, technique_lexicon_by_language.csv")

setup,model,model+lexicon,delta
language,,,
english,0.6259,0.6234,-0.0025
singlish,0.6483,0.6345,-0.0138
sinhala,0.6207,0.6164,-0.0043
tamil,0.6207,0.6069,-0.0138
tamilish,0.5714,0.5676,-0.0039


wrote technique_lexicon_ablation.csv, technique_lexicon_by_language.csv


## 5. Verdict

Read the delta against the CI width, not against zero. Dev holds 68 unique Negative tickets, so
the interval is roughly ±0.08 — a gain smaller than that is not measurable here regardless of
sign.

If the gain is null or negative, the conclusion is **not** "lexicon correction does not work" —
it is that *a lexicon mined from the training data* adds nothing on top of a model already fit to
that data, which is the predicted outcome stated at the top. Testing Senevirathna's actual claim
needs an externally authored banking-polarity lexicon, which is a labeling task. Record that as
the follow-up rather than as a refutation.